# FT-01 : Introduction au Fine-Tuning

**Objectif** : Comprendre les trois approches de fine-tuning (complet, partiel, LoRA) et mettre en pratique LoRA sur un petit modele de texte.

**Plan** :
1. Pourquoi fine-tuner ?
2. Full Fine-Tuning vs Partial vs LoRA
3. Comparaison des parametres entrainables
4. LoRA en pratique : fine-tuning de GPT-2 sur un style litteraire
5. Evaluation avant/apres

**Materiel requis** : GPU avec ~4 Go VRAM (GPT-2 = 124M params).

In [1]:
import torch
import os
import gc

print(f"PyTorch {torch.__version__}")
print(f"CUDA disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go")

PyTorch 2.11.0+cu126
CUDA disponible : True
GPU : NVIDIA GeForce RTX 3090
VRAM : 25.8 Go


## 1. Pourquoi fine-tuner ?

Les modeles pre-entraines (foundation models) sont capables de taches generales, mais manquent de specialisation :

| Cas d'usage | Pourquoi fine-tuner ? |
|-------------|----------------------|
| Style d'ecriture specifique | Le modele generique ne maitrise pas un style particulier |
| Domaine technique (medical, juridique) | Le vocabulaire et les schemas de raisonnement different |
| Tache structuree (classification, extraction) | Le format de sortie n'est pas naturel pour le modele |
| Reduction des hallucinations | Un modele specialise fait moins d'erreurs dans son domaine |

**Exemple** : GPT-2 genere du texte anglais generique. Apres fine-tuning sur des textes francais du 19e siecle, il peut imiter ce style specifique.

## 2. Trois approches de fine-tuning

### 2a. Full Fine-Tuning

Tous les parametres du modele sont mis a jour.

- **Avantage** : Performance maximale, le modele s'adapte completement
- **Inconvenient** : Tres couteux en GPU (VRAM = modeles + gradients + optimizer states, souvent 3-4x la taille du modele)
- **Usage** : Modeles petits (<1B) ou budgets GPU importants

### 2b. Partial Fine-Tuning (Feature Extraction)

On gele les couches inferieures et n'entraine que les dernieres.

- **Avantage** : Moins de parametres a entrainer, plus rapide
- **Inconvenient** : Adaptation limitee, pas de modification des representations profondes
- **Usage** : Quand le domaine d'application est proche du pre-entrainement

### 2c. LoRA (Low-Rank Adaptation)

On injecte de petites matrices de rang faible ($A \times B$) a cote des poids originaux, et on n'entraine que ces matrices.

- **Avantage** : Tres peu de parametres entrainables (0.1-1% du modele), stockage minimal
- **Inconvenient** : Performances legerement inferieures au full fine-tuning sur des taches complexes
- **Usage** : Standard industriel actuel, surtout pour modeles >3B

### Formulation mathematique de LoRA

Pour un poids pre-entraine $W \in \mathbb{R}^{d \times k}$ :

$$h = Wx + \Delta W x = Wx + BAx$$

ou $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times k}$, et $r \ll \min(d, k)$.

Le rang $r$ (typiquement 4-64) controle le compromis expressivite/efficacite.

## 3. Comparaison des parametres entrainables

Chargeons un petit modele (GPT-2, 124M params) et comparons les trois approches.

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "openai-community/gpt2"  # 124M params

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
total_params = sum(p.numel() for p in model.parameters())
print(f"Modele : {MODEL_NAME}")
print(f"Parametres totaux : {total_params:,} ({total_params/1e6:.1f}M)")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Modele : openai-community/gpt2
Parametres totaux : 124,439,808 (124.4M)


In [3]:
def count_trainable(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

def freeze_all(model):
    for p in model.parameters():
        p.requires_grad = False

def unfreeze_last_n_layers(model, n):
    freeze_all(model)
    for block in model.transformer.h[-n:]:
        for p in block.parameters():
            p.requires_grad = True
    for p in model.lm_head.parameters():
        p.requires_grad = True

# Approche 1 : Full fine-tuning
for p in model.parameters():
    p.requires_grad = True
full_trainable, total = count_trainable(model)

# Approche 2 : Partial (2 dernieres couches + lm_head)
unfreeze_last_n_layers(model, 2)
partial_trainable, _ = count_trainable(model)

# Approche 3 : LoRA
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],  # Attention projections dans GPT-2
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Recharger le modele propre pour LoRA
model_lora = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model_lora = get_peft_model(model_lora, lora_config)
lora_trainable, _ = count_trainable(model_lora)

print(f"{'Approche':<20} {'Entrainables':>15} {'Total':>15} {'%':>8}")
print(f"{'-'*20} {'-'*15} {'-'*15} {'-'*8}")
print(f"{'Full Fine-Tuning':<20} {full_trainable:>15,} {total:>15,} {100.0:>7.2f}%")
print(f"{'Partial (2 couches)':<20} {partial_trainable:>15,} {total:>15,} {partial_trainable/total*100:>7.2f}%")
print(f"{'LoRA (r=8)':<20} {lora_trainable:>15,} {total:>15,} {lora_trainable/total*100:>7.2f}%")
print(f"\nRatio LoRA/Full = {lora_trainable/full_trainable*100:.2f}%")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

W0524 03:44:00.962000 33800 torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Approche                Entrainables           Total        %
-------------------- --------------- --------------- --------
Full Fine-Tuning         124,439,808     124,439,808  100.00%
Partial (2 couches)       52,773,120     124,439,808   42.41%
LoRA (r=8)                   294,912     124,439,808    0.24%

Ratio LoRA/Full = 0.24%


C:\Users\jsboi\AppData\Roaming\Python\Python313\site-packages\peft\tuners\lora\layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


## 4. LoRA en pratique : Fine-tuning sur un style litteraire

Nous allons fine-tuner GPT-2 pour generer du texte dans le style de Victor Hugo.
Le dataset est un extrait des *Miserables* (domaine public).

In [4]:
from datasets import Dataset

# Extrait des Miserables (Victor Hugo, domaine public)
miserables_text = """Cosette, enentrant dans la vieille maison de la rue de l'Homme-Arme, 
n'avait pas pu quitter la main du vieuxhomme. Cosette regardait la chambre 
obscure, et la figure ridée de son bienfaiteur, et cette figure ridée était 
l'immense figure de la misere humaine.

Jean Valjean etait devenu le bonhomme de la maison. Il avait pris l'habitude 
de descendre a la cave pour rapporter du vin, et de monter au grenier pour 
rapporter du bois. Cosette trouvait que monsieur Jean avait les mains pleines 
de bienfaits.

La nuit, quand tout dormait dans la maison, Jean Valjean restait eveille. 
Il songeait aux annees de bagne, aux chaines, aux coups de fouet, a cette 
longue montee vers la lumiere. Cosette dormait, et il la regardait dormir.

Les rues de Paris etaient sombres en ce temps-la. Les reverberes jetaient 
des taches de lumiere jaune sur les pavés mouilles. Jean Valjean marchait 
vite, comme un homme qui fuit quelque chose, ou qui cherche quelqu'un.

Marius l'observait de loin. Ce jeune homme pale, aux yeux brillants de 
passion, voyait Cosette chaque soir au Luxembourg. Il n'osait lui parler. 
L'amour est timide, meme pour les coeurs les plus courageux.

La barricade s'elevait dans la rue de la Grange-aux-Belles. Les jeunes 
hommes avaient pris les pavés et construit un mur de pierres. Enjolras, 
beau comme un dieu, commandait avec une autorite naturelle. Grantaire, 
ivre et fidele, le suivait partout.

Gavroche passait entre les balles comme un oiseau. Ce gamin de Paris, 
nenfant abandonne devenu enfant de la patrie, ramassait les cartouches 
des soldats morts pour les rapporter aux insurgés. Son chant retentissait 
dans la nuit, clair et courageux."""

# Decouper en segments d'entrainement
segments = [s.strip() for s in miserables_text.split("\n\n") if s.strip()]
print(f"Segments d'entrainement : {len(segments)}")
print(f"Exemple : {segments[0][:100]}...")

# Creer le dataset
train_data = Dataset.from_dict({"text": segments})
print(f"Dataset : {len(train_data)} exemples")

Segments d'entrainement : 7
Exemple : Cosette, enentrant dans la vieille maison de la rue de l'Homme-Arme, 
n'avait pas pu quitter la main...
Dataset : 7 exemples


In [5]:
# Tokenisation
def tokenize_function(examples):
    result = tokenizer(
        examples["text"],
        truncation=True,
        max_length=256,
        padding="max_length",
    )
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_dataset = train_data.map(tokenize_function, batched=True, remove_columns=["text"])
print(f"Tokens par exemple : {len(tokenized_dataset[0]['input_ids'])}")

Map:   0%|          | 0/7 [00:00<?, ? examples/s]

Tokens par exemple : 256


In [6]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Configuration LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model_to_train = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model_to_train = get_peft_model(model_to_train, lora_config)
model_to_train.print_trainable_parameters()

# Arguments d'entrainement
training_args = TrainingArguments(
    output_dir="./temp_ft01_output",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    learning_rate=5e-4,
    weight_decay=0.01,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=42,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = Trainer(
    model=model_to_train,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print(f"\nEntrainement en cours...")
train_result = trainer.train()
print(f"\nPerte finale : {train_result.training_loss:.4f}")
print(f"Temps : {train_result.metrics['train_runtime']:.1f}s")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

C:\Users\jsboi\AppData\Roaming\Python\Python313\site-packages\peft\tuners\lora\layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364



Entrainement en cours...


C:\Users\jsboi\AppData\Roaming\Python\Python313\site-packages\torch\nn\parallel\data_parallel.py:45: UserWarning: 
    There is an imbalance between your GPUs. You may want to exclude GPU 1 which
    has less than 75% of the memory or cores of GPU 0. You can do so by setting
    the device_ids argument to DataParallel, or by setting the CUDA_VISIBLE_DEVICES
    environment variable.
  if warn_imbalance(lambda props: props.total_memory):


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


C:\Users\jsboi\AppData\Roaming\Python\Python313\site-packages\torch\nn\parallel\comm.py:109: UserWarning: PyTorch is not compiled with NCCL support
  if nccl.is_available(inputs):


Step,Training Loss
5,5.168762
10,5.062768
15,4.959981
20,4.850943



Perte finale : 5.0106
Temps : 13.3s


## 5. Evaluation avant / apres

Comparons la generation du modele original vs le modele fine-tune.

In [7]:
# Modele original (recharger)
model_original = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model_original.eval()

def generate_text(model, prompt, max_new_tokens=80, temperature=0.8):
    inputs = tokenizer(prompt, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

prompt = "Dans les rues sombres de Paris,"

print("=" * 60)
print("GENERATION ORIGINALE (GPT-2 pre-entraine)")
print("=" * 60)
original_output = generate_text(model_original, prompt)
print(original_output)

print("\n" + "=" * 60)
print("GENERATION APRES LoRA (fine-tune style Hugo)")
print("=" * 60)
model_to_train.eval()
finetuned_output = generate_text(model_to_train, prompt)
print(finetuned_output)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GENERATION ORIGINALE (GPT-2 pre-entraine)


Dans les rues sombres de Paris, l'amour d'un seul et de l'accérieur en France, l'un peut-être la monde en France, l'ambassador sombre, l'équipage, l'éclaration des consommateurs en France, l'équipage, l'équipage d'une développ

GENERATION APRES LoRA (fine-tune style Hugo)


Dans les rues sombres de Paris, les rues à la nouvelle à la lune.

(1) Le rue de Paris.

(2) Le rue de Paris.

(3) Les rues dans les lèves.

(4) Le rue de Paris.

(5) Le rue de Paris.

(6) Les r


In [8]:
# Second prompt pour comparer
prompt2 = "Jean Valjean marchait dans la nuit"

print("=" * 60)
print(f"Prompt : \"{prompt2}\"")
print("=" * 60)

print("\nORIGINAL :")
print(generate_text(model_original, prompt2))

print("\nFINE-TUNE :")
print(generate_text(model_to_train, prompt2))

Prompt : "Jean Valjean marchait dans la nuit"

ORIGINAL :


Jean Valjean marchait dans la nuit (I want you to see)

It's the moment when the truth is revealed.

(A) Vigour de vivre, je vous répondent qu'on avait, avec le journée à ce soir à l'hôtel, je vous vous avez pas avait.

(B) Vig

FINE-TUNE :


Jean Valjean marchait dans la nuit à la vie est réponse une réputation des fois ne pas de la voiture, il vie en vie.

Jusqu'il avait selon à la même de la plus d'une faire qui seulement un sommes la plus d'une faire.

Je vous les présentants dans la


## 6. Taille des adaptateurs LoRA

Un des avantages majeurs de LoRA : les poids appris sont minuscules.

In [9]:
import tempfile

# Sauvegarder l'adaptateur LoRA
adapter_dir = os.path.join(tempfile.gettempdir(), "ft01_lora_adapter")
model_to_train.save_pretrained(adapter_dir)

# Taille totale du modele original vs adaptateur
adapter_size = sum(
    os.path.getsize(os.path.join(adapter_dir, f))
    for f in os.listdir(adapter_dir)
    if f.endswith(".safetensors") or f.endswith(".bin")
)

model_size_mb = total_params * 4 / 1e6  # float32
adapter_size_mb = adapter_size / 1e6

print(f"Modele complet (GPT-2) : {model_size_mb:.1f} Mo")
print(f"Adaptateur LoRA seul  : {adapter_size_mb:.2f} Mo")
print(f"Ratio adaptateur/modele : {adapter_size_mb/model_size_mb*100:.2f}%")
print(f"\nPour deployer : il suffit du modele de base + adaptateur ({adapter_size_mb:.1f} Mo)")

Modele complet (GPT-2) : 497.8 Mo
Adaptateur LoRA seul  : 1.18 Mo
Ratio adaptateur/modele : 0.24%

Pour deployer : il suffit du modele de base + adaptateur (1.2 Mo)


## 7. Nettoyage memoire GPU

In [10]:
del model_original, model_to_train, trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"VRAM libre : {torch.cuda.mem_get_info()[0] / 1e9:.1f} Go")
else:
    print("Nettoyage termine")

VRAM libre : 24.4 Go


## 8. Exercices

Appliquez les concepts de ce notebook.

In [11]:
# Exercice 1 : Exploration du rang LoRA
# TODO etudiant : Creez 3 configurations LoRA avec r=2, r=8, r=32
# et comparez le nombre de parametres entrainables pour chacune.
# Quel rang donne le meilleur ratio performance/taille ?

print("Exercice a completer : comparez les rangs LoRA r=2, 8, 32")

Exercice a completer : comparez les rangs LoRA r=2, 8, 32


In [12]:
# Exercice 2 : Fine-tuning sur un autre style
# TODO etudiant : Remplacez le dataset des Miserables par un extrait
# d'un autre auteur (ex: Baudelaire, Verlaine) et relancez l'entrainement.
# Observez comment le style de generation change.

print("Exercice a completer : fine-tunez sur un style different")

Exercice a completer : fine-tunez sur un style different


In [13]:
# Exercice 3 : Impact du learning rate
# TODO etudiant : Entrainnez avec learning_rate = 1e-5, 5e-4, 1e-3.
# Observez la perte finale et la qualite de generation.
# Que se passe-t-il avec un learning rate trop eleve ?

print("Exercice a completer : comparez les learning rates")

Exercice a completer : comparez les learning rates


## Resume

| Approche | Params entrainables | VRAM requise | Stockage adaptateur |
|----------|--------------------:|-------------:|--------------------:|
| Full Fine-Tuning | 100% | 3-4x taille modele | Modele complet |
| Partial (2 couches) | ~5-15% | ~1.5x taille modele | Couches modifiees |
| **LoRA (r=8)** | **~0.5%** | **~1.1x taille modele** | **~2-5 Mo** |

**Points cles** :
- LoRA decompose la mise a jour en matrices de bas rang $B \times A$ ou $r \ll d, k$
- Seuls ~0.5% des parametres sont entraines, mais la qualite reste proche du full fine-tuning
- Les adaptateurs sont portables : un seul modele de base + N adaptateurs pour N taches
- Le rang $r$ controle le compromis expressivite/efficacite

**Prochaines etapes** (FT-02) : LoRA sur un modele plus grand avec quantization 4-bit (QLoRA).